In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor
)
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
from pathlib import Path
candidates = [
    Path("../data/PV_Diagnostic_Cleaned.csv"),
    Path("data/PV_Diagnostic_Cleaned.csv"),
    Path("../explore&clean/PV_Diagnostic_Cleaned.csv"),
    Path("explore&clean/PV_Diagnostic_Cleaned.csv"),
    Path("PV_Diagnostic_Cleaned.csv"),
]

dataset_path = next((p for p in candidates if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Could not find PV_Diagnostic_Cleaned.csv in standard paths.")

print(f"Loading dataset from: {dataset_path}")
df = pd.read_csv(dataset_path, sep=";")

# Filter daytime records (Irr >= 50 W/m2)
day_df = df[df['Irr'] >= 50.0].copy().reset_index(drop=True)
print(f"Total daytime samples: {len(day_df):,} out of {len(df):,}")

Loading dataset from: ..\data\PV_Diagnostic_Cleaned.csv
Total daytime samples: 221,190 out of 226,175


In [3]:
df_normal = day_df[day_df['Label_S1'] == 'Normal'].copy()

# 2. Select features and target
data = df_normal[['Irr', 'T1', 'P1']].dropna()

X = data[['Irr', 'T1']]
y = data['P1']

# 3. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [4]:
models = {
    'Linear Regression': make_pipeline(
        StandardScaler(),
        LinearRegression()
    ),

    'Ridge Regression': make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    ),

    'Random Forest': RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    'Extra Trees': ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    'Hist Gradient Boosting': HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        random_state=42
    ),

    'XGBoost': XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=1.0,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    )
}

# 5. Train and evaluate models
results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred)
    })

In [5]:

# 6. Compare results
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('RMSE').reset_index(drop=True)

print(results_df.round(4).to_string(index=False))

# 7. Best model based on RMSE
best_model_name = results_df.loc[0, 'Model']
best_model = models[best_model_name]

print("\nBest model:", best_model_name)

# 8. Actual vs predicted values
comparison = pd.DataFrame({
    'Actual_P1': y_test,
    'Predicted_P1': predictions[best_model_name]
}, index=y_test.index)

print(comparison.head(10))

                 Model     MAE    RMSE     R2
           Extra Trees  5.5140 16.1061 0.7619
         Random Forest  6.0815 16.2055 0.7589
Hist Gradient Boosting 10.0062 18.4836 0.6864
               XGBoost 11.4177 19.6883 0.6442
     Gradient Boosting 13.3977 21.1031 0.5912
     Linear Regression 21.4423 28.5895 0.2497
      Ridge Regression 21.4423 28.5895 0.2497

Best model: Extra Trees
        Actual_P1  Predicted_P1
54725    0.049438      0.046826
138326  56.253675     56.255399
149768   0.043320      0.015332
165817  46.685283     47.345461
141689  72.464757     60.982346
177646  72.662344     61.158309
219278  18.350917     18.133146
17493    0.331439      4.054658
177854  71.230072     71.295820
149593   0.027574      0.015266


In [6]:
print("P1 minimum:", y.min())
print("P1 maximum:", y.max())
print("P1 mean:", y.mean())
print("P1 standard deviation:", y.std())



P1 minimum: 0.0
P1 maximum: 146.947265625
P1 mean: 41.6867369561857
P1 standard deviation: 32.886074179264156
